# Santander Customer Transaction Prediction TC1 Human Baseline

This notebook defines the TC1 from-scratch human baseline for the **santander-customer-transaction-prediction** competition.
Each preprocessing section is tagged with its `CA-XXXXXX` action ID from `candidate_actions.json`.

## Load Data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.home() / '.cache' / 'kagglehub' / 'competitions' / 'santander-customer-transaction-prediction'

required_files = ['train.csv', 'test.csv']
missing = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files under {DATA_DIR}: {missing}')

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

FEATURES = [c for c in train.columns if c.startswith('var_')]

print('train shape:', train.shape)
print('test shape: ', test.shape)
print(f'feature cols: {len(FEATURES)}')
print('target distribution:', train['target'].value_counts().to_dict())
train.head(2)

## CA-000002 — Drop ID_code Column

`ID_code` is a unique string row identifier with no predictive relationship
to the transaction target. It must be excluded from the feature matrix to
avoid identifier memorisation.

Store test IDs separately for submission construction.

In [ ]:
# CA-000002: DROP_COLUMNS (ID_code)
test_ids = test['ID_code'].copy()

train = train.drop(columns=['ID_code'])
test  = test.drop(columns=['ID_code'])

y_train = train['target'].values
train   = train.drop(columns=['target'])

print('train features shape:', train.shape)
print('test features shape: ', test.shape)
print('positive class rate: ', y_train.mean())

## CA-000009 — Value-Count Features

For each of the 200 `var_*` features, append a new column counting how many
times each value appears in the combined **train + test** rows.

In [ ]:
# CA-000009: APPLY_EXPRESSION (value-count features from train + test)
df_all = pd.concat([train[FEATURES], test[FEATURES]], axis=0).reset_index(drop=True)

for col in FEATURES:
    vc = df_all[col].value_counts()
    train[col + '_vc'] = train[col].map(vc).fillna(0).astype(int)
    test[col + '_vc']  = test[col].map(vc).fillna(0).astype(int)

print(f'Added {len(FEATURES)} value-count columns.')
print('train shape after VC features:', train.shape)
print('test shape after VC features: ', test.shape)
train[[FEATURES[0], FEATURES[0] + '_vc']].head(3)

## Final Shape Check

In [ ]:
print('Final train shape:', train.shape)
print('Final test shape: ', test.shape)
print('y_train shape:    ', y_train.shape)

assert train.shape[1] == test.shape[1], 'Column mismatch between train and test'
assert len(train) == len(y_train), 'Row mismatch between features and target'
print('All shape checks passed.')
train.head(2)